# JSON Structure Analysis for Simulation Results

This notebook analyzes the structure of JSON result files from the simulation infrastructure and orchestrator.

Target directory: `src/notebooks/data/results_sim/125-225-5clients-knative-with-network/`


In [1]:
import json
import pandas as pd
import numpy as np
from pathlib import Path
import pprint
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns

# Configure display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

# Pretty printer for JSON structure
pp = pprint.PrettyPrinter(indent=2, width=120)


## 1. Load JSON Files and Basic Structure Analysis


In [2]:
# Define paths to the JSON files
data_dir = Path('data/results_sim/125-225-5clients-knative-with-network')
json_files = list(data_dir.glob('*.json'))

print(f"Found {len(json_files)} JSON files:")
for file in json_files:
    print(f"  - {file.name} ({file.stat().st_size / (1024**2):.1f} MB)")


Found 2 JSON files:
  - simulation_2.json (926.5 MB)
  - simulation_1.json (497.1 MB)


In [3]:
def load_json_file(filepath):
    """Load a JSON file and return the data"""
    print(f"Loading {filepath.name}...")
    with open(filepath, 'r') as f:
        data = json.load(f)
    print(f"✓ Loaded {filepath.name} successfully")
    return data

# Load the first file for analysis
first_file = json_files[0]
simulation_data = load_json_file(first_file)


Loading simulation_2.json...
✓ Loaded simulation_2.json successfully


## 2. Top-Level Structure Analysis


In [4]:
def get_all_keys(data, parent_key='', start_path=None, max_depth=None, current_depth=0):
    keys = []

    # Handle start_path only at top level
    if start_path and not parent_key:
        for key in start_path.split('.'):
            if isinstance(data, dict) and key in data:
                data = data[key]
            else:
                raise KeyError(f"Key path '{start_path}' not found in the JSON.")
        parent_key = start_path
        current_depth = 0  # reset depth since we're starting here

    # Stop if max depth is reached
    if max_depth is not None and current_depth >= max_depth:
        keys.append(parent_key)
        return keys

    if isinstance(data, dict):
        for k, v in data.items():
            full_key = f"{parent_key}.{k}" if parent_key else k
            keys.extend(get_all_keys(v, full_key, max_depth=max_depth, current_depth=current_depth + 1))
    elif isinstance(data, list):
        for i, item in enumerate(data):
            full_key = f"{parent_key}[{i}]"
            keys.extend(get_all_keys(item, full_key, max_depth=max_depth, current_depth=current_depth + 1))
    else:
        keys.append(parent_key)

    return keys


In [16]:
# Get all keys starting from 'stats.systemStateResults'
keys = get_all_keys(simulation_data["stats"]["taskResults"][0], start_path="executionPlatform", max_depth=1)

for key in keys:
    print(key)

executionPlatform


In [17]:
simulation_data["stats"]["taskResults"][0]["executionPlatform"]

'xavierCpu'

In [7]:
def flatten_json(y, parent_key='', sep='.'):
    items = []

    if isinstance(y, dict):
        for k, v in y.items():
            new_key = f"{parent_key}{sep}{k}" if parent_key else k
            items.extend(flatten_json(v, new_key, sep=sep).items())
    elif isinstance(y, list):
        for i, v in enumerate(y):
            new_key = f"{parent_key}[{i}]"
            items.extend(flatten_json(v, new_key, sep=sep).items())
    else:
        items.append((parent_key, y))

    return dict(items)

In [8]:
import pandas as pd
import json

# Case 1: data is a list of objects
if isinstance(simulation_data, list):
    flat_data = [flatten_json(item) for item in simulation_data]

# Case 2: data has nested logs (e.g., stats.taskResults)
elif "stats" in simulation_data and "taskResults" in simulation_data["stats"]:
    flat_data = [flatten_json(item) for item in simulation_data["stats"]["taskResults"]]
else:
    flat_data = [flatten_json(simulation_data)]  # Just flatten the top-level JSON

df = pd.DataFrame(flat_data)

In [9]:
df

,taskId,dispatchedTime,scheduledTime,arrivedTime,startedTime,doneTime,applicationType.name,taskType.name,taskType.platforms[0],taskType.platforms[1],taskType.platforms[2],taskType.memoryRequirements.rpiCpu,taskType.memoryRequirements.xavierGpu,taskType.memoryRequirements.xavierCpu,taskType.coldStartDuration.rpiCpu,taskType.coldStartDuration.xavierGpu,taskType.coldStartDuration.xavierCpu,taskType.executionTime.rpiCpu,taskType.executionTime.xavierGpu,taskType.executionTime.xavierCpu,taskType.energy.rpiCpu,taskType.energy.xavierGpu,taskType.energy.xavierCpu,taskType.imageSize.rpiCpu,taskType.imageSize.xavierGpu,taskType.imageSize.xavierCpu,taskType.stateSize.ae-dnn2.input,taskType.stateSize.ae-dnn2.output,taskType.stateSize.es-dnn2.input,taskType.stateSize.es-dnn2.output,taskType.stateSize.nofs-dnn1.input,taskType.stateSize.nofs-dnn1.output,platform.shortName,platform.name,platform.hardware,platform.price,platform.idleEnergy,elapsedTime,pullTime,coldStartTime,executionTime,waitTime,queueTime,initializationTime,computeTime,communicationsTime,coldStarted,cacheHit,localDependencies,localCommunications,energy,networkLatency,sourceNode,executionNode,executionPlatform,gnn_decision_time,taskType.stateSize.nofs-dnn2.input,taskType.stateSize.nofs-dnn2.output
0,0,0.018071,0.018071,37.424664,37.520664,37.565659,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.194450e-08,3.057,2.99,2.987,20000,8000,12800,8000,153600.0,8000.0,xavierCpu,Nvidia Xavier Carmel Arm v8.2,cpu,3399,0.000785,37.547589,37.323664,0.096,0.023918,0.0,37.406593,0.096,0.044995,0.021078,True,False,True,False,8.194450e-08,0.101,node0,node6,xavierCpu,0.0,NaN,NaN
1,1,0.018071,0.018071,37.666659,37.666659,37.711655,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.194450e-08,3.057,2.99,2.987,20000,8000,12800,8000,153600.0,8000.0,xavierCpu,Nvidia Xavier Carmel Arm v8.2,cpu,3399,0.000785,37.693584,0.000000,0.000,0.023918,0.0,37.648589,0.000,0.044995,0.021078,False,True,True,False,8.194450e-08,0.101,node0,node6,xavierCpu,0.0,NaN,NaN
2,2,0.018071,0.018071,37.812655,37.812655,37.857650,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.194450e-08,3.057,2.99,2.987,20000,8000,12800,8000,153600.0,8000.0,xavierCpu,Nvidia Xavier Carmel Arm v8.2,cpu,3399,0.000785,37.839579,0.000000,0.000,0.023918,0.0,37.794584,0.000,0.044995,0.021078,False,True,True,False,8.194450e-08,0.101,node0,node6,xavierCpu,0.0,NaN,NaN
3,3,0.018071,0.018071,37.958650,37.958650,38.003645,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.194450e-08,3.057,2.99,2.987,20000,8000,12800,8000,153600.0,8000.0,xavierCpu,Nvidia Xavier Carmel Arm v8.2,cpu,3399,0.000785,37.985575,0.000000,0.000,0.023918,0.0,37.940579,0.000,0.044995,0.021078,False,True,True,False,8.194450e-08,0.101,node0,node6,xavierCpu,0.0,NaN,NaN
4,4,0.018071,0.018071,38.104645,38.104645,38.149641,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.194450e-08,3.057,2.99,2.987,20000,8000,12800,8000,153600.0,8000.0,xavierCpu,Nvidia Xavier Carmel Arm v8.2,cpu,3399,0.000785,38.131570,0.000000,0.000,0.023918,0.0,38.086575,0.000,0.044995,0.021078,False,True,True,False,8.194450e-08,0.101,node0,node6,xavierCpu,0.0,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
271402,271402,243.802540,243.802540,730.635097,730.635097,730.680092,nofs-dnn1,dnn1,rpiCpu,xavierGpu,xavierCpu,0.069229,0.9,0.068974,0.636,6.14,0.096,0.16842,0.036249,0.023918,1.357640e-07,1.416670e-07,8.19